# 00. Research question, sources, and current GAVD inventory

We will define normal-only anomaly detection, distinguish it from supervised condition recognition, and count the current annotations before training anything.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

## 1. Define the unit of evidence

A source video may contain several walking sequences. A sequence may provide many overlapping windows. These windows share a person, camera, and recording conditions. The experiment therefore splits source video IDs before building windows. A source ID still does not guarantee a unique participant.

The five labels are `normal`, `parkinsons`, `stroke`, `cerebral palsy`, and `myopathic`. Repeated mentions of stroke create no extra class. Myopathic gait is principally associated with muscle disease, so the paper should describe a neurological and neuromuscular cohort rather than imply that every label has the same neurological cause.

![Research tasks](images/research_map.svg)

In [ ]:
summary=manifest.groupby('label').agg(sequences=('sequence_id','size'), annotation_rows=('n_frames','sum'), source_videos=('source_id','nunique')).reindex(LABELS)
display(summary)
print('Selected rows:', manifest.n_frames.sum())
from aaai_gait.data import manifest_digest
print('Fresh annotation SHA256:', manifest_digest(manifest))

## 2. Understand the starting architecture

Skeleton S-JEPA predicts teacher features at hidden joint-time tokens. An online encoder sees visible tokens. A predictor receives those context features and learned positional queries. A slowly updated target encoder sees the complete clean skeleton and supplies stopped-gradient targets.

Our adaptation retains centered and sharpened target distributions with cross-entropy. It changes masking to uniform sampling, adds VICReg, uses a small laptop model, and pools the online encoder for downstream classification. The original paper uses a different benchmark setup and target-encoder readout. We cannot claim an exact reproduction.

MAMP predicts motion targets rather than teacher latent distributions. Its published default favors motion, while its official code also contains a uniform random masking option. StoP adds positional noise. The 2026 stochastic-latent JEPA learns a prior and posterior over latent variables. These are three distinct ideas.

![Randomness](images/stochasticity.svg)

## 3. Inspect source separation

The splitter uses label and source counts only. It never searches for a split that produces a favorable model score. Freeze these assignments before extraction and record every exclusion afterwards. Current class proportions differ from a clinic's prevalence, so a probability learned here should not be interpreted as a clinical risk.

In [ ]:
display(manifest.groupby(['split','label']).agg(sequences=('sequence_id','size'),sources=('source_id','nunique')))
assert manifest.groupby('source_id').split.nunique().max() == 1

## 4. See why more data can still leave uncertainty

The current subset contains 666 sequences, 140,641 annotation rows, and 103 source IDs. Only 11 source videos contribute Parkinson's examples and 11 contribute cerebral-palsy examples. Frames increase temporal detail but do not create independent people. The README explains missing participant metadata, recording bias, and the need for an independent cohort.

![Generalizability](images/generalization.svg)

The [ASTAD4 site](https://ismart.ece.mcgill.ca/astad4/) lists November 20, 2026 as the deadline and permits seven content pages plus two reference pages. A successful submission needs measured evidence and careful limits, not just a working model.

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.